In [1]:
!pip install scann sentence-transformers datasets
!pip uninstall --yes tensorflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 3.9 MB/s eta 0:00:00
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0


# **Generate embedding**

In [10]:
import pandas as pd
import h5py
from sentence_transformers import SentenceTransformer

# Setup
CSV_FILE = 'problem_resolution.csv'
H5_OUTPUT = 'problem.h5'

# 1. Load CSV
df = pd.read_csv(CSV_FILE)
headlines = df['issue_description'].astype(str).tolist() # Ensure your column name is correct

# 2. Generate Embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(headlines, show_progress_bar=True, normalize_embeddings=True)

# 3. Save only vectors to H5
with h5py.File(H5_OUTPUT, 'w') as hf:
    hf.create_dataset("embeddings", data=embeddings, compression="gzip")

print(f"Saved {len(embeddings)} embeddings. CSV will be used for text retrieval.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

Saved 39970 embeddings. CSV will be used for text retrieval.


# **Load embedding**

In [13]:
import csv
import h5py
import scann
from sentence_transformers import SentenceTransformer

def run_search_from_csv(query_text, k=5):
    # 1. Load the original CSV for text retrieval using built-in csv
    # We load it into a list so we can access specific rows by index
    data_rows = []
    with open('problem_resolution.csv', mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            data_rows.append(row)

    # 2. Load embeddings from H5
    with h5py.File('problem.h5', 'r') as hf:
        dataset_embeddings = hf['embeddings'][:]

    # 3. Build ScaNN
    searcher = scann.scann_ops_pybind.builder(
        dataset_embeddings, k, "dot_product"
    ).score_brute_force().build()

    # 4. Embed Query
    model = SentenceTransformer('all-MiniLM-L6-v2')
    query_vec = model.encode([query_text], normalize_embeddings=True)[0]

    # 5. Search
    indices, distances = searcher.search(query_vec, final_num_neighbors=k)

    # 6. Retrieve text from the list using the indices
    print(f"\nQuery: {query_text}\n" + "-"*30)

    for idx, dist in zip(indices, distances):
        # Access the row directly from our list
        row = data_rows[idx]
        print(f"[{dist:.4f}]\nProblem: {row['issue_description']} \nResolution: {row['resolution_notes']}")

# Usage
run_search_from_csv("sign in")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: sign in
------------------------------
[0.4079] Problem: I am unable to access my account after entering the correct credentials. 
Resolutioin:Bug logged internally and workaround shared with the customer.
[0.4079] Problem: I am unable to access my account after entering the correct credentials. 
Resolutioin:Subscription status corrected and confirmation email sent to customer.
[0.4079] Problem: I am unable to access my account after entering the correct credentials. 
Resolutioin:Security settings updated and customer notified of precautionary measures.
[0.4079] Problem: I am unable to access my account after entering the correct credentials. 
Resolutioin:Explained billing breakdown and clarified applicable charges.
[0.4079] Problem: I am unable to access my account after entering the correct credentials. 
Resolutioin:Explained billing breakdown and clarified applicable charges.
